In [1]:
# Import Gemini SDK
from google import genai
from google.genai import types
import time

# Initialize Gemini client with your API key
client = genai.Client(api_key="AQ.Ab8RN6L2ebaIjbAqbmvTfEHkmNAwnSdwpKNvzOs_wf9SwNGbmA")

In [13]:
def get_completion_from_messages(messages, temperature=0, max_tokens=500):
    system_instruction = None
    conversation = []

    for msg in messages:
        if msg["role"] == "system":
            system_instruction = msg["content"]
        else:
            role = "model" if msg["role"] == "assistant" else "user"
            conversation.append({
                "role": role,
                "parts": [{"text": msg["content"]}]
            })

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="models/gemini-2.5-flash",
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=temperature,
                    max_output_tokens=max_tokens,
                ),
                contents=conversation
            )
            # Guard against None response
            if response and response.text:
                return response.text.strip()
            else:
                print(f"Attempt {attempt+1}: Empty response received, retrying...")
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(3)

    return "Sorry, the server is unavailable. Please try again later."

In [14]:
# Gemini has no dedicated moderation endpoint like OpenAI
# Instead, we prompt Gemini to act as a content moderator
# and return a structured JSON analysis of the content

final_response_to_customer = f"""
The SmartX ProPhone has a 6.1-inch display, 128GB storage, \
12MP dual camera, and 5G. The FotoSnap DSLR Camera \
has a 24.2MP sensor, 1080p video, 3-inch LCD, and \
interchangeable lenses. We have a variety of TVs, including \
the CineView 4K TV with a 55-inch display, 4K resolution, \
HDR, and smart TV features. We also have the SoundMax \
Home Theater system with 5.1 channel, 1000W output, wireless \
subwoofer, and Bluetooth. Do you have any specific questions \
about these products or any other products we offer?
"""

# Send the content to Gemini for moderation analysis
moderation_messages = [
    {
        "role": "system",
        "content": """You are a content moderation assistant. 
        Analyze the input text and return a JSON with these fields:
        - harmful (true/false)
        - categories: object with these boolean fields:
            - harassment
            - hate
            - violence
            - sexual
            - self_harm
            - dangerous_content
        - reason (brief explanation of your decision)"""
    },
    {
        "role": "user",
        "content": final_response_to_customer
    }
]

moderation_output = get_completion_from_messages(moderation_messages)
print("── Moderation Output ──")
print(moderation_output)

── Moderation Output ──
```json
{
  "harmful": false,
  "categories": {
    "harassment": false,
    "hate": false,
    "violence": false,
    "sexual": false,
    "self_harm": false,
    "dangerous_content": false
  },
  "reason": "The text describes electronic products and their specifications, and asks if the user has questions. It contains no harmful content."
}
```


In [15]:
# System message instructs Gemini to act as an evaluator
# It checks two things:
# 1. Whether the agent's response correctly uses product information
# 2. Whether the response sufficiently answers the customer's question
# Output is a single character: Y or N
system_message = f"""
You are an assistant that evaluates whether \
customer service agent responses sufficiently \
answer customer questions, and also validates that \
all the facts the assistant cites from the product \
information are correct.
The product information and user and customer \
service agent messages will be delimited by \
3 backticks, i.e. ```.
Respond with a Y or N character, with no punctuation:
Y - if the output sufficiently answers the question \
AND the response correctly uses product information
N - otherwise

Output a single letter only.
"""

# The original customer query being evaluated
customer_message = f"""
tell me about the smartx pro phone and \
the fotosnap camera, the dslr one. \
Also tell me about your tvs"""

# Product catalog used as ground truth for fact-checking
product_information = """{ "name": "SmartX ProPhone", "category": "Smartphones and Accessories", "brand": "SmartX", "model_number": "SX-PP10", "warranty": "1 year", "rating": 4.6, "features": [ "6.1-inch display", "128GB storage", "12MP dual camera", "5G" ], "description": "A powerful smartphone with advanced camera features.", "price": 899.99 } { "name": "FotoSnap DSLR Camera", "category": "Cameras and Camcorders", "brand": "FotoSnap", "model_number": "FS-DSLR200", "warranty": "1 year", "rating": 4.7, "features": [ "24.2MP sensor", "1080p video", "3-inch LCD", "Interchangeable lenses" ], "description": "Capture stunning photos and videos with this versatile DSLR camera.", "price": 599.99 } { "name": "CineView 4K TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-4K55", "warranty": "2 years", "rating": 4.8, "features": [ "55-inch display", "4K resolution", "HDR", "Smart TV" ], "description": "A stunning 4K TV with vibrant colors and smart features.", "price": 599.99 } { "name": "SoundMax Home Theater", "category": "Televisions and Home Theater Systems", "brand": "SoundMax", "model_number": "SM-HT100", "warranty": "1 year", "rating": 4.4, "features": [ "5.1 channel", "1000W output", "Wireless subwoofer", "Bluetooth" ], "description": "A powerful home theater system for an immersive audio experience.", "price": 399.99 } { "name": "CineView 8K TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-8K65", "warranty": "2 years", "rating": 4.9, "features": [ "65-inch display", "8K resolution", "HDR", "Smart TV" ], "description": "Experience the future of television with this stunning 8K TV.", "price": 2999.99 } { "name": "SoundMax Soundbar", "category": "Televisions and Home Theater Systems", "brand": "SoundMax", "model_number": "SM-SB50", "warranty": "1 year", "rating": 4.3, "features": [ "2.1 channel", "300W output", "Wireless subwoofer", "Bluetooth" ], "description": "Upgrade your TV's audio with this sleek and powerful soundbar.", "price": 199.99 } { "name": "CineView OLED TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-OLED55", "warranty": "2 years", "rating": 4.7, "features": [ "55-inch display", "4K resolution", "HDR", "Smart TV" ], "description": "Experience true blacks and vibrant colors with this OLED TV.", "price": 1499.99 }"""

# The agent's response that needs to be evaluated
# final_response_to_customer should already be defined from a previous cell
q_a_pair = f"""
Customer message: ```{customer_message}```
Product information: ```{product_information}```
Agent response: ```{final_response_to_customer}```

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""

# Send to Gemini for evaluation
# max_tokens=5 ensures only a single character (Y or N) is returned
messages = [
    {"role": "system", "content": system_message},
    {"role": "user",   "content": q_a_pair}
]
# Increase max_tokens from 1 to 5 to allow Gemini to return Y or N
response = get_completion_from_messages(messages, max_tokens=5)
# Strip whitespace and take only the first character to ensure single Y or N
result = response.strip()[0] if response else "Error"
print("── Evaluation Result (Y = Pass, N = Fail) ──")
print(response)

── Evaluation Result (Y = Pass, N = Fail) ──
N


In [16]:
another_response = "life is like a box of chocolates"
q_a_pair = f"""
Customer message: ```{customer_message}```
Product information: ```{product_information}```
Agent response: ```{another_response}```

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""
messages = [
    {'role': 'system', 'content': system_message},
    {'role': 'user', 'content': q_a_pair}
]

response = get_completion_from_messages(messages)
print(response)

N
